# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# バックアップと復元

**取れているかどうかは、取れなくなって初めて分かる。** 数字で確かめ、月に一度は実際に開く。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. 全体像

控えは**2か所**で取っている。役割が違う。

| | いつ | 置き場 | 世代 | 守れるもの |
|---|---|---|---|---|
| **ホストの cron** | 毎日 2:40 | ホストの `/opt/kosenmap/backups/` | 14 | 操作ミス・データの壊れ(細かく戻せる) |
| **この PC のタスク** | 毎週日曜 3:00 | この PC の `D:\Backups\<ホスト>-<日時>\` | 7 | **VPS ごと失われたとき** |

**錠は2種類。**

| もの | 錠 | 開ける鍵 |
|---|---|---|
| 控えそのもの(`.tar.gz.cms`) | **証明書で暗号化**(ホストに置くのは公開鍵だけ) | **この PC の秘密鍵**(`%USERPROFILE%\.kosenmap`)。**ホストは自分の控えを開けない** |
| メールに添える実行記録(`.tar.gz.enc`) | **合言葉**(ホストの `.env` の `BACKUP_PASSPHRASE`) | 合言葉。**メール本文には書かない** |

中身は **MariaDB・Postgres(Logto)のダンプ、`uploads/`、`.env`、`config/*.local.php`** ——
復元に要るものは、そのまま乗っ取りに要るもの。**平文で置かない・送らない。**

手元の置き場を決めているのは `backup-lib.ps1` の `Get-KmBackupRoot`(既定 `D:\Backups`。
`%USERPROFILE%\.kosenmap\backup-root.txt` に書けば変えられる。**置き場が無ければ止まる**)。

## 2. 鍵(一度だけ)

### 鍵の状態を見る

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\backup-keys.ps1 -Check

### 鍵を作る

**普段は要らない(作ってある)。** 作り直すと**それまでの控えが開けなくなる**ので、`-Force` は本当に作り直すときだけ。
作ると証明書をホストへ置く(`-SkipUpload` で置かない)。

```powershell
.\backup-keys.ps1            # 作って、ホストへ証明書を置く
.\backup-keys.ps1 -SkipUpload
```

**秘密鍵(`%USERPROFILE%\.kosenmap`)を失うと、どの控えも開けない。** この PC の外にも控えを持つこと。

## 3. 取る

### いま控えを取る(この PC へ)

ホストで作って暗号化し、この PC へ持ってきて検査する。**MariaDB のダンプの末尾に `Dump completed` があるか、
テーブル数・行数**まで見て `summary.txt` に残す。**「ファイルができた」を成功と見なさない。**

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番で控えを作り、この PC へ持ってきます(ホストの backups/ に1世代増えます)"
.\backup-data.ps1

### 同じことを配備スクリプトから

配備はせず、取るだけ。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番で控えを作り、この PC へ持ってきます"
.\deploy-to-host.ps1 -BackupOnly -Yes

## 4. 開く

### 何があるかだけ見る

開かない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\open-backup.ps1 -List

### 一番新しいものを開いて検査する

**検査が済んだら平文は消える。** 月に一度はこれを流して「開ける」ことを確かめる。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
.\open-backup.ps1

### 開いたまま残す(復元に使うとき)

```powershell
.\open-backup.ps1 -Path 'D:\Backups\ito8795.com-20260913-085636\km-backup-ubuntu-20260913-085635.tar.gz.cms' -Keep
```

⚠ **残るのは平文**(`.env` と `config/*.local.php` 入り)。`<置き場>\_opened\<日時>` に置かれ、
**24時間たつと世代整理が消す**。使い終わったらすぐ消すこと。

## 5. 週次タスク(この PC)

タスクスケジューラの「KosenMap バックアップ(週次)」が、毎週日曜 3:00 に `backup-task-run.ps1` を呼ぶ。
**失敗したら、ホスト経由のメールと Windows の通知**で知らせる(回線やホストが落ちているとメールは出せないので、通知が要る)。
**PC の電源が入っていない週はタスク自体が走らない** —— そのときはホストの日曜の便りが「手元の PC が今週取りに来ていません」と言う。

**繋ぎ方(2026-09-18 から):** タスクは `-User kmops -KeyPath ~\.ssh\km_backup -Gated` で走る。
`km_backup` は**控え専用の鍵**で、ホスト側の門番(`scripts/ssh-backup-gate.sh`)が
**控えを作る・取る・疎通を見る・失敗を知らせる**以外を断る(シェルも sudo も docker も転送も通らない)。
無人で動く鍵なので**パスフレーズは付けられない** —— その代わりに門番で狭めてある([12](12-hardening-2026-09-15.ipynb) §7-4 A)。
**鍵やホストを変えたら、必ず `register-backup-task.ps1` で登録し直す**(タスクには引数が焼き付いているので、既定を変えても効かない)。


### 何が登録されるかだけ見る

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\register-backup-task.ps1 -WhatIfOnly

### 登録し直す

曜日や時刻を変えるときは `-DayOfWeek Monday -At 04:30` など。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
.\register-backup-task.ps1

### いま一度走らせる

記録は `<置き場>\backup-task.log`。数分かかる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "週次タスクをいま走らせます(本番で控えを作り、この PC へ持ってきます)"
Start-ScheduledTask -TaskName 'KosenMap バックアップ(週次)'
"始めました。記録: $(. .\backup-lib.ps1; Get-KmBackupRoot)\backup-task.log"

### 失敗の知らせを試す

**取得だけをわざと失敗させ、知らせは本番ホスト経由で届ける。** 件名に `【試験】` が付く。
`no-such-host.invalid` は「存在しないことが決まっている名前」なので、どこへも繋がらない。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "試験の失敗メールを1通送ります"
$root = Join-Path $env:TEMP 'kosenmap-mailtest'
New-Item -ItemType Directory -Force $root | Out-Null
# 週次タスクと同じ形(門番付きの鍵)で送る。取得は no-such-host.invalid で必ず失敗する
.\backup-task-run.ps1 -HostName no-such-host.invalid -MailHostName ito4.jp -Gated -KeyPath "$env:USERPROFILE\.ssh\km_backup" -NoToast -BackupRoot $root
"(失敗させる試験なので、終了コード 1 が正しい)"

## 6. ホスト側の控え

### いま在るもの

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
./scripts/host-backup.sh --path /opt/kosenmap --list

### 日曜と同じ便り(実行記録の添付つき)を出す

**root で走らせる**(実行記録 `/var/log/kosenmap` と、添付の置き場は root でないと扱えない)。
`kmops` の権限(`%%host`)で走らせると、控えは取れるが添付は付かない。1世代増える。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal --confirm "本番で控えを作り、実行記録を暗号化して添付した便りを送ります(sudo のパスワードを聞かれます)"
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-backup.sh --path /opt/kosenmap --notify --heartbeat --attach-logs"

## 7. メールに添えられた実行記録を開く

届いた `kosenmap-logs-<日時>.tar.gz.enc` を保存して、下のセルの `$enc` をそのパスに書き換えて実行する。
**合言葉はホストの `.env` から openssl へ直接流す**(画面にも履歴にも出さない)。
ホスト(OpenSSL 3.0.13)で閉じたものを、この PC(OpenSSL 4.x)で開けることは確かめてある。

### 合言葉(`BACKUP_PASSPHRASE`)の決まり

- **英数字だけにする。** `"` `'` `\` `$` `` ` `` を入れると、docker compose が `.env` 全体を読めなくなり、
  **DB も取れず、メールも1通も出ない**(2026-09-13 に実際に起きた)
- 変えたら、**それ以前の添付は新しい合言葉では開けない**(控えそのもの `.cms` は証明書なので関係ない)

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
# .env は置き場の持ち主 kmops だけが読める(2026-09-18 から。docs/12 §7-4 B 段 2)
# ↓ 保存した添付のパスに書き換える
$enc = "$env:USERPROFILE\Downloads\kosenmap-logs-XXXXXXXX-XXXXXX.tar.gz.enc"
if (-not (Test-Path $enc)) { throw "ファイルがありません: $enc" }
$out = Join-Path (Split-Path $enc) 'kosenmap-logs-opened'
New-Item -ItemType Directory -Force $out | Out-Null
ssh -i "$env:USERPROFILE\.ssh\km_ops" -o BatchMode=yes kmops@ito4.jp "grep -s '^BACKUP_PASSPHRASE=' /opt/kosenmap/.env | head -n 1 | cut -d= -f2-" |
    openssl enc -d -aes-256-cbc -pbkdf2 -in $enc -out "$out\logs.tar.gz" -pass stdin
if ($LASTEXITCODE -ne 0) { throw '復号できませんでした(合言葉を変える前に作られた添付かもしれません)' }
tar -xzf "$out\logs.tar.gz" -C $out
Get-ChildItem $out -Recurse -File | Select-Object @{n='ファイル'; e={ $_.FullName.Replace("$out\", '') }}, Length
"見終わったら消してください: Remove-Item -Recurse -Force '$out'"

## 8. 戻す(復元)

`Old/restore-data.ps1` は 2026-09-14 に直した。**本番と切り離した使い捨ての環境へ実際に全部戻し、
行数がすべて一致することを確かめてある**(地点 1037 / 経路 988 / MariaDB 31 表 / Postgres 79 表 / uploads)。
コンテナは compose の札(`-Project`、既定 `kosenmap`)で探し、平文のダンプはホストの一時フォルダ(700)に置いて必ず消す。

### 流れ

1. 控えを開いて残す(§4 の `-Keep`)。中の `km-mariadb.sql` / `km-postgres.sql` / `km-uploads.tar.gz` を使う
2. **まず `-DryRun`**(転送と検査だけ。書き込まない。期待される行数が出る)
3. 本番の復元。**移行先に既にテーブルがあると中断する**(上書きは `-Force`。その場合も現状を `km-before-<日時>.sql` に退避する)
4. **行数を突き合わせる**(スクリプト自身が、ダンプから読んだ期待値と移行先の数を比べ、合わなければ例外で止まる)
5. `.env` と `config/*.local.php` は**自動では戻さない**(人が置く)

```powershell
..\Old\restore-data.ps1 -BackupDir 'D:\Backups\_opened\<日時>' -HostName <ホスト> -DryRun
..\Old\restore-data.ps1 -BackupDir 'D:\Backups\_opened\<日時>' -HostName <ホスト>
```

### 一番の落とし穴: `POSTGRES_USER` は名前まで同じにする

`pg_dump` は所有者をロール名で書く。移行先にそのロールが無いと `role "Main" does not exist` で落ち、
**Logto の DB が空のまま残る**。ロールは Postgres が**データ領域が空の初回起動のときだけ**作るので、
**`.env` に同じ `POSTGRES_USER` を書いてから初めて `up -d` する** → Logto が起動して `logto_tenant_logto` を作るのを待つ → 復元。

### 戻したあとに数字で確かめる

| 見るもの | 確かめ方 |
|---|---|
| 地図のノード・経路 | `SELECT COUNT(*) FROM km_map_nodes` / `km_map_edges` |
| 教職員氏名の件数 | `SELECT COUNT(*) FROM km_map_nodes WHERE occupant_name IS NOT NULL` |
| 問い合わせ・監査ログ | `km_form_submissions` / `SELECT COUNT(*), MAX(created_at) FROM km_admin_log` |
| uploads の実体 | 台帳(`km_files` など)と実体の数が釣り合うか |
| 時刻 | 監査ログの最新が**日本時間**か |

画面でも: 地図が出る・教職員氏名の解除が効く・**ファイルが実際にダウンロードできる**・受信箱に過去の問い合わせ・**管理画面にサインインできる**。

経緯と細部は [../Old/docs/data-migration.md](../Old/docs/data-migration.md)。